In [23]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [24]:
import random
import torch


# 랜덤시드 고정
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

In [25]:
import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import ImageFolder
from tqdm import tqdm
from PIL import Image

# 0. 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
rootpath = '/kaggle/input/2026-rcv-winter-urp-cnn/landmark_datasets_18011747/'
csv_path = os.path.join(rootpath, 'submit.csv')

# 1. 전처리 설정
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 2. 데이터셋 정의 (Train)
train_datasets = ImageFolder(root=os.path.join(rootpath, 'train'), transform=transform)
label_map = train_datasets.class_to_idx 
inv_label_map = {v: k for k, v in label_map.items()} 

# 3. 테스트 데이터셋 정의 (파일명 정렬 문제 해결)
class TestDataset(Dataset):
    def __init__(self, data_path, csv_path, transform=None):
        self.data_path = data_path
        self.transform = transform
        # csv 파일의 순서대로 파일 리스트를 생성하여 정렬 뒤섞임 방지
        df = pd.read_csv(csv_path)
        self.img_list = df['id'].tolist() # csv의 파일명 컬럼명이 'id'인지 확인 필요

    def __len__(self):
        return len(self.img_list)

    def __getitem__(self, idx):
        img_name = self.img_list[idx]
        
        # 확장자 중복 방지 및 대소문자 대응
        # 이미 확장자가 있다면 그대로 쓰고, 없으면 .jpg를 붙임
        if not (img_name.lower().endswith('.jpg') or img_name.lower().endswith('.jpeg') or img_name.lower().endswith('.png')):
            img_path = os.path.join(self.data_path, img_name + '.jpg')
        else:
            img_path = os.path.join(self.data_path, img_name)
            
        # 만약 여기서 에러가 난다면 경로 문제일 확률이 높으므로 시도해봅니다.
        if not os.path.exists(img_path):
            # 경로를 'test/test/파일명'으로 재시도 (캐글 특유의 구조 대응)
            retry_path = os.path.join(self.data_path, 'test', img_name)
            if os.path.exists(retry_path):
                img_path = retry_path

        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image

test_datasets = TestDataset(
    data_path=os.path.join(rootpath, 'test'), 
    csv_path=csv_path, 
    transform=transform
)

# 4. DataLoader
train_loader = DataLoader(dataset=train_datasets, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(dataset=test_datasets, batch_size=64, shuffle=False, num_workers=2)

# 5. CNN 모델 설계
class CNN(nn.Module): 
    def __init__(self, num_classes):
        super(CNN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        # 224 -> 112 -> 56 -> 28
        self.fc = nn.Sequential(
            nn.Linear(128 * 28 * 28, 512),
            nn.ReLU(),
            nn.Dropout(0.5), # 과적합 방지
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# 6. 모델 및 최적화 설정
model = CNN(num_classes=len(label_map)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_func = nn.CrossEntropyLoss()

# 7. 학습 루프
epochs = 5
model.train()
for epoch in range(epochs):
    epoch_loss, epoch_acc = 0, 0
    for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        out = model(x)
        cost = loss_func(out, y)
        cost.backward()
        optimizer.step()

        epoch_loss += cost.item()
        epoch_acc += (torch.argmax(out, dim=1) == y).sum().item()
        
    print(f'Loss: {epoch_loss/len(train_loader):.4f} | Acc: {epoch_acc/len(train_datasets)*100:.2f}%')

# 8. 예측 루프
model.eval()
pred_indices = []
with torch.no_grad():
    for x in tqdm(test_loader, desc="Predicting"):
        x = x.to(device)
        out = model(x)
        predict = torch.argmax(out, dim=1)
        pred_indices.extend(predict.cpu().numpy())


Epoch 1: 100%|██████████| 47/47 [00:46<00:00,  1.01it/s]


Loss: 1.5035 | Acc: 48.77%


Epoch 2: 100%|██████████| 47/47 [00:45<00:00,  1.02it/s]


Loss: 0.6559 | Acc: 79.23%


Epoch 3: 100%|██████████| 47/47 [00:45<00:00,  1.02it/s]


...

Epoch 4: 100%|██████████| 47/47 [00:46<00:00,  1.00it/s]


Loss: 0.2520 | Acc: 92.07%


Epoch 5: 100%|██████████| 47/47 [00:47<00:00,  1.00s/it]


Loss: 0.1406 | Acc: 95.80%


Predicting: 100%|██████████| 16/16 [00:15<00:00,  1.01it/s]


In [26]:
final_pred_labels = [inv_label_map[idx] for idx in pred_indices]

submit = pd.read_csv(csv_path)
submit['label'] = final_pred_labels # CSV의 레이블 컬럼명에 맞게 수정
submit.to_csv('submission.csv', index=False)

In [28]:
print(submit)

                         id          label
0    HF030010_0401_0028.jpg       imjingak
1    HF030001_0401_0271.JPG     building63
2    HF030005_0201_0180.jpg         btower
3    HF030005_0201_1299.jpg         btower
4    HF030001_0601_0512.JPG        soonsin
..                      ...            ...
995  HF030012_0000_0490.jpg         gtower
996  HF030009_0201_1331.jpg         skview
997  HF030011_0201_2815.jpg  incheonbridge
998  HF030005_0201_0336.jpg         btower
999  HF030005_0201_0120.jpg         btower

[1000 rows x 2 columns]
--Call--
> /usr/local/lib/python3.12/dist-packages/IPython/core/displayhook.py(252)__call__()
    250         sys.stdout.flush()
    251 
--> 252     def __call__(self, result=None):
    253         """Printing with history cache management.
    254 



ipdb>  submit


*** NameError: name 'submit' is not defined


ipdb>  q
